# Lab 03 - Cloud Evaluation (starter)

Complete the `TODO` blocks. Reference: `lab03_cloud_evaluation_solution.ipynb`.

A Foundry project (`FOUNDRY_PROJECT_ENDPOINT`) is mandatory for this lab.

## Step 0 - Configuration (ready to run)

In [ ]:
import os, sys, json, time, warnings
from pprint import pprint

from lab_utils import load_settings

warnings.filterwarnings("ignore")

TO_BE_EVALUATED_FILE = "./assets/synthetic_dataset_cloud.jsonl"
FILE_VERSION = "1.0"

settings = load_settings(verbose=True)
credential = settings["credential"]
foundry_project_endpoint = settings["foundry_project_endpoint"]
judge_deployment = settings["azure_evaluation_compatible_deployment_name"]

if not foundry_project_endpoint:
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT is required for cloud evaluation")

In [ ]:
from azure.ai.projects import AIProjectClient

# TODO 0.1 - create the AIProjectClient and get its OpenAI client
project_client = ...
openai_client = ...

## Step 1 - Inspect the dataset (~5 min)

Look at the four fields of `assets/synthetic_dataset_cloud.jsonl`: they must match the schema you
declare in Step 4.

In [ ]:
# TODO 1.1 - load the JSONL and print the number of records and the keys of the first one

## Step 2 - Helper functions (ready to run)

Dataset versions are immutable, so uploading the same version twice fails: this helper validates the
file and finds the first usable version.

In [ ]:
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError


def increase_version(version: str) -> str:
    """Return the next version by incrementing its numeric final part."""
    parts = version.split(".")
    if len(parts) > 1:
        return f"{'.'.join(parts[:-1])}.{int(parts[-1]) + 1}"
    return str(int(version) + 1)


def validate_jsonl(file_path: str) -> None:
    """Fail early on malformed datasets: the cloud run would fail much later."""
    with open(file_path, encoding="utf-8") as jsonl_file:
        for line_number, line in enumerate(jsonl_file, start=1):
            if not line.strip():
                raise ValueError(f"Blank line found in {file_path} at line {line_number}")
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {file_path} at line {line_number}, column {error.colno}: {error.msg}"
                ) from error
            if not isinstance(record, dict):
                raise ValueError(f"Expected a JSON object in {file_path} at line {line_number}")


def upload_dataset(project_client, file_path: str, initial_version: str,
                   max_version_attempts: int = 20, listing_checks: int = 12,
                   listing_check_interval: int = 5):
    """Upload a dataset using the first usable version that appears in the project listing."""
    if file_path.lower().endswith(".jsonl"):
        validate_jsonl(file_path)

    file_name = file_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]
    file_version = initial_version

    for _ in range(max_version_attempts):
        version_is_listed = any(
            d.name == file_name and d.version == file_version for d in project_client.datasets.list()
        )
        try:
            project_client.datasets.get(name=file_name, version=file_version)
            version_exists = True
        except ResourceNotFoundError:
            version_exists = False

        if version_is_listed:
            print(f"Dataset {file_name} version {file_version} is already listed; trying the next version")
            file_version = increase_version(file_version)
            continue

        if version_exists:
            project_client.datasets.delete(name=file_name, version=file_version)
            print(f"Deletion requested for unlisted dataset {file_name} version {file_version}")
            file_version = increase_version(file_version)
            continue

        try:
            dataset = project_client.datasets.upload_file(
                name=file_name, file_path=file_path, version=file_version
            )
        except ResourceExistsError:
            print(f"Dataset {file_name} version {file_version} appeared during upload; trying the next version")
            file_version = increase_version(file_version)
            continue

        for _ in range(listing_checks):
            if any(d.name == dataset.name and d.version == dataset.version
                   for d in project_client.datasets.list()):
                return dataset
            time.sleep(listing_check_interval)

        raise RuntimeError(
            f"Dataset {dataset.name} version {dataset.version} was uploaded but did not appear in the "
            f"project listing after {listing_checks * listing_check_interval} seconds."
        )

    raise RuntimeError(f"No usable version found for {file_name} after {max_version_attempts} attempts")

## Step 3 - Upload the dataset (~5 min)

In [ ]:
# TODO 3.1 - call upload_dataset(project_client=..., file_path=TO_BE_EVALUATED_FILE, initial_version=FILE_VERSION)
file_dataset = ...
data_id = ...
print(data_id)

## Step 4 - Data source config and testing criteria (~15 min)

Declare the item schema, then one criterion per evaluator. Start with `builtin.f1_score`
(deterministic, no model) and only afterwards add the AI judges.

Mapping syntax: `"response": "{{item.response}}"`.

In [ ]:
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator

# TODO 4.1 - DataSourceConfigCustom with query / response / context / ground_truth
data_source_config = ...

# TODO 4.2 - one criterion for builtin.f1_score (response + ground_truth, no model)
# TODO 4.3 - add builtin.groundedness and builtin.relevance
#            (initialization_parameters={"model": judge_deployment})
# TODO 4.4 - add builtin.violence
testing_criteria = [...]

## Step 5 - Create the evaluation and the run (~10 min)

In [ ]:
# TODO 5.1 - openai_client.evals.create(name=..., data_source_config=..., testing_criteria=...)
eval_object = ...

# TODO 5.2 - openai_client.evals.runs.create(eval_id=..., name=..., data_source=CreateEvalJSONLRunDataSourceParam(
#            type="jsonl", source=SourceFileID(type="file_id", id=data_id)))
eval_run = ...
print(eval_run.id, eval_run.status)

## Step 6 - Poll and read the results (~10 min)

In [ ]:
# TODO 6.1 - loop on openai_client.evals.runs.retrieve(...) until the status is
#            completed / failed / canceled, sleeping 5 seconds between checks
# TODO 6.2 - print eval_run.report_url and open it in the portal
# TODO 6.3 - list the output items and print the per-criterion scores of the first rows

### Checkpoint - everything below is optional

## Step 7 (optional) - Add the custom evaluators published in Lab 01

Reference them by `evaluator_name` and `evaluator_version`, and pass the
`initialization_parameters` declared at publish time
(`deployment_name` + `threshold` for friendliness, `deployment_name` + `pass_threshold` for
response length). Check the actual version numbers in the Foundry portal.

In [ ]:
# TODO 7.1 - extend testing_criteria with friendliness_evaluator and response_length_score_evaluator
# TODO 7.2 - create a new evaluation + run and compare the report with the previous one

## Step 8 (optional) - Evaluate the dataset you generated in Lab 02

Reshape `02-dataset-generation/generated_datasets/simulated_conversations.jsonl` into the
`query` / `context` / `response` / `ground_truth` schema and run the same evaluation on it.

In [ ]:
# TODO 8.1 - flatten the simulated conversations into user/assistant pairs
# TODO 8.2 - save the new JSONL, upload it and repeat steps 5-6